# Vertex AI AutoML Tabular Inference (Online & Batch)

This tutorial demonstrates model serving and inference options for AutoML Tabular models using `tabflows`.

### Objectives
1. Discover trained AutoML Tabular `aiplatform.Model` resources from Vertex AI Model Registry or a completed `PipelineJob`.
2. Deploy the model to a real-time `aiplatform.Endpoint` and execute Online Inference.
3. Submit a Batch Prediction job against a GCS dataset and retrieve output predictions.
4. Clean up deployed endpoint resources to manage serving costs.

In [13]:
import os

from dotenv import load_dotenv
from google.cloud import aiplatform

from tabflows import (
    TabularPipelineConfig,
    cleanup_endpoint,
    deploy_model_to_endpoint,
    list_models,
    predict_online,
    run_batch_prediction,
)

# Load environment variables from local .env file
load_dotenv()
print("Environment and libraries loaded successfully.")

Environment and libraries loaded successfully.


In [14]:
# TabularPipelineConfig automatically loads GCP_PROJECT, GCP_LOCATION, GCP_BUCKET_URI
config = TabularPipelineConfig()

print(f"Project ID: {config.project_id}")
print(f"Location: {config.location}")
print(f"Serving Machine Type: {config.serving_machine_type}")

Project ID: hybrid-vertex
Location: us-central1
Serving Machine Type: n1-standard-4


In [15]:
# Option 1: Discover recent models automatically from Vertex AI Model Registry
print("\n--- Discovering Recent Models in Vertex AI ---")
try:
    all_models = list_models(config=config, limit=10)
    tabular_models = [
        m
        for m in all_models
        if "tabular" in m.display_name.lower() or "automl" in m.display_name.lower()
    ]
    candidate_models = tabular_models if tabular_models else all_models

    if candidate_models:
        print(f"Found {len(candidate_models)} candidate model(s):")
        for idx, m in enumerate(candidate_models):
            print(f"  [{idx}] Display Name: {m.display_name}")
            print(f"      Resource Name: {m.resource_name}")

        model = candidate_models[0]
        print(f"\nSelected Model: {model.resource_name}")
    else:
        print("No models found in Vertex AI Model Registry.")
except Exception as e:
    print(f"Could not list models automatically: {e}")

# Option 2: Alternatively specify MODEL_RESOURCE_NAME in your .env or variable below
if "model" not in locals() or not isinstance(model, aiplatform.Model):
    MODEL_RESOURCE_NAME = os.getenv("MODEL_RESOURCE_NAME", "")
    if MODEL_RESOURCE_NAME:
        model = aiplatform.Model(model_name=MODEL_RESOURCE_NAME)
        print(f"Loaded Vertex AI Model: {model.resource_name}")
    else:
        print("Notice: Provide a valid 'model' object or set MODEL_RESOURCE_NAME in .env")


--- Discovering Recent Models in Vertex AI ---
Found 1 candidate model(s):
  [0] Display Name: tabular-workflow-model-7e373ee4-6f03-48f7-b23d-a3b6e30b6dbf
      Resource Name: projects/934903580331/locations/us-central1/models/612894719357222912

Selected Model: projects/934903580331/locations/us-central1/models/612894719357222912


## 1. Real-Time Online Inference

Deploy the model to a real-time endpoint (`n1-standard-4`), send prediction payloads, and undeploy the endpoint.

In [16]:
# Set flag to True to trigger real-time endpoint deployment & online prediction
RUN_ONLINE_INFERENCE = False

# Optional: Specify an existing deployed Endpoint resource name to predict instantly:
# e.g., "projects/YOUR_PROJECT/locations/us-central1/endpoints/ENDPOINT_ID"
EXISTING_ENDPOINT_RESOURCE_NAME = ""

sample_instance = {
    "age": "35",
    "job": "technician",
    "marital": "married",
    "education": "tertiary",
    "default": "no",
    "balance": "1350",
    "housing": "yes",
    "loan": "no",
    "contact": "cellular",
    "day": "15",
    "month": "may",
    "duration": "220",
    "campaign": "1",
    "pdays": "-1",
    "previous": "0",
    "poutcome": "unknown",
}

print("Sample Feature Instance:", sample_instance)

if RUN_ONLINE_INFERENCE:
    if EXISTING_ENDPOINT_RESOURCE_NAME:
        print(f"Predicting against existing endpoint '{EXISTING_ENDPOINT_RESOURCE_NAME}'...")
        predictions = predict_online(
            endpoint=EXISTING_ENDPOINT_RESOURCE_NAME,
            instances=[sample_instance],
        )
        print("Online Predictions Output:", predictions)
    elif "model" in locals() and isinstance(model, aiplatform.Model):
        print(f"Deploying model '{model.resource_name}' to endpoint (takes ~10-15 mins)...")
        endpoint = deploy_model_to_endpoint(model=model, config=config, sync=True)
        predictions = predict_online(endpoint=endpoint, instances=[sample_instance])
        print("Online Predictions Output:", predictions)
        print("Cleaning up deployed endpoint...")
        cleanup_endpoint(endpoint=endpoint)
        print("Endpoint cleaned up successfully.")
    else:
        print("Error: No valid 'model' object found to deploy endpoint.")
else:
    print("RUN_ONLINE_INFERENCE is set to False. Set RUN_ONLINE_INFERENCE = True to run.")

Sample Feature Instance: {'age': '35', 'job': 'technician', 'marital': 'married', 'education': 'tertiary', 'default': 'no', 'balance': '1350', 'housing': 'yes', 'loan': 'no', 'contact': 'cellular', 'day': '15', 'month': 'may', 'duration': '220', 'campaign': '1', 'pdays': '-1', 'previous': '0', 'poutcome': 'unknown'}
RUN_ONLINE_INFERENCE is set to False. Set RUN_ONLINE_INFERENCE = True to run.


## 2. Batch Inference

Submit a non-blocking batch prediction job for large offline datasets stored in Cloud Storage or BigQuery.

In [17]:
# Set flag to True to submit a Batch Prediction job to Vertex AI
RUN_BATCH_INFERENCE = True

gcs_input_csv = f"{config.bucket_uri}/test_instances.csv"
gcs_output_prefix = f"{config.root_dir}/batch_predictions"

print(f"Input GCS Source: {gcs_input_csv}")
print(f"Output GCS Destination Prefix: {gcs_output_prefix}")

Input GCS Source: gs://jts-tabflows-v1/test_instances.csv
Output GCS Destination Prefix: gs://jts-tabflows-v1/automl_tabular_pipeline/batch_predictions


In [ ]:
if RUN_BATCH_INFERENCE:
    if "model" in locals() and isinstance(model, aiplatform.Model):
        print(f"Submitting Batch Prediction job for model '{model.resource_name}'...")
        batch_job = run_batch_prediction(
            model=model,
            config=config,
            gcs_source=gcs_input_csv,
            gcs_destination_prefix=gcs_output_prefix,
        )
        print(f"Batch Prediction Job submitted successfully: {batch_job.resource_name}")
    else:
        print("Error: No valid 'model' object found to submit batch job.")
else:
    print("RUN_BATCH_INFERENCE is set to False. Set RUN_BATCH_INFERENCE = True to run.")

Submitting Batch Prediction job for model 'projects/934903580331/locations/us-central1/models/612894719357222912'...
Creating BatchPredictionJob


RuntimeError: BatchPredictionJob resource has not been created.

BatchPredictionJob created. Resource name: projects/934903580331/locations/us-central1/batchPredictionJobs/3069614097113808896
To use this BatchPredictionJob in another session:
bpj = aiplatform.BatchPredictionJob('projects/934903580331/locations/us-central1/batchPredictionJobs/3069614097113808896')
View Batch Prediction Job:
https://console.cloud.google.com/agent-platform/locations/us-central1/batch-predictions/3069614097113808896?project=934903580331
BatchPredictionJob projects/934903580331/locations/us-central1/batchPredictionJobs/3069614097113808896 current state:
JOB_STATE_RUNNING
BatchPredictionJob projects/934903580331/locations/us-central1/batchPredictionJobs/3069614097113808896 current state:
JOB_STATE_RUNNING
BatchPredictionJob projects/934903580331/locations/us-central1/batchPredictionJobs/3069614097113808896 current state:
JOB_STATE_RUNNING
